# 04 -- VPP Economics

**Purpose (PROJECT.md Section 8.7):** physics-to-GBP translation, DNO reinforcement avoidance, battery-equivalence framing, retrofit cost-benefit. Week 3.

**Scope boundary (PROJECT.md Section 8.1):** this is illustrative placeholder economics (flexibility-service price, retrofit cost, battery-hardware comparison), not a merchant dispatch or market-mechanics model. Whether a payment mechanism exists for this specific flexibility product at residential scale is FORWARD-LOOKING and unresolved -- every output that touches revenue says so.

**No-double-counting (PROJECT.md Section 2.3 / Stage E):** apply DNO/VPP value only on top of the physical peak-reduction result from Notebooks 01-03. Never double-count against comfort-floor behaviour already inside the coastdown model.


In [1]:
import sys

print("Python executable:", sys.executable)
assert "thermal-counterfactual-gb" in sys.executable, (
    "Wrong kernel selected -- pick the 'thermal-counterfactual-gb' kernel, "
    "not a default/global one. Run setup.sh first if it doesn't exist yet."
)

import numpy as np
import polars as pl
import scipy
import matplotlib
print("polars:", pl.__version__, "| numpy:", np.__version__, "| scipy:", scipy.__version__)


Python executable: /tmp/kernelenv/thermal-counterfactual-gb/bin/python3


polars: 1.43.2 | numpy: 2.2.6 | scipy: 1.15.3


## 1. Load assumptions and Notebook 02/03 output

In [2]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("../src").resolve()))
import yaml
import polars as pl
import numpy as np

CONFIG_PATH = Path("../configs/tenure_insulation_assumptions.yml")
with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

population_03 = pl.read_parquet("../data/intermediate/03_estate_population_model.parquet")
sim_02 = pl.read_parquet("../data/intermediate/02_cold_snap_simulation.parquet")

comfort = cfg["comfort_band"]
cop = cfg["heat_pump"]["cop_at_cold_snap"]
c_kwh_per_k = cfg["thermal_capacity"]["c_kwh_per_k"]

print(population_03)


shape: (3, 9)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ scenario  ┆ kw_per_ho ┆ weighted_ ┆ mc_avoide ┆ … ┆ mc_avoide ┆ feeder_in ┆ feeder_in ┆ stress_t │
│ ---       ┆ me_during ┆ insulatio ┆ d_kw_p50  ┆   ┆ d_kw_p90  ┆ sulation_ ┆ sulation_ ┆ est_brea │
│ str       ┆ _peak     ┆ n_prevale ┆ ---       ┆   ┆ ---       ┆ prevalenc ┆ prevalenc ┆ ch_delta │
│           ┆ ---       ┆ nce       ┆ f64       ┆   ┆ f64       ┆ e_p…      ┆ e_w…      ┆ _c       │
│           ┆ f64       ┆ ---       ┆           ┆   ┆           ┆ ---       ┆ ---       ┆ ---      │
│           ┆           ┆ f64       ┆           ┆   ┆           ┆ f64       ┆ f64       ┆ i32      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ status_qu ┆ 2.028806  ┆ 0.2703    ┆ 1.867537  ┆ … ┆ 2.299703  ┆ 0.266667  ┆ 0.0       ┆ -8       │
│ o_no_vpp  ┆           ┆           ┆           ┆   ┆           ┆           ┆

## 2. Retrofit cost-benefit

Use `retrofit_cost_gbp_to_epc_c` by tenure from config -- do not invent a blended figure without showing the weighting.


In [3]:
tenure_mix = cfg["tenure_mix"]
retrofit_cost = cfg["retrofit_cost_gbp_to_epc_c"]

# tenure_mix and retrofit_cost_gbp_to_epc_c use different key names for the
# same four segments -- map explicitly rather than guess a convention.
tenure_to_cost_key = {
    "local_authority_retained": "local_authority",
    "housing_association_retained": "housing_association",
    "ex_rtb_privately_rented": "private_rented",
    "ex_rtb_owner_occupied": "owner_occupied",
}

weighted_cost_per_home = sum(
    tenure_mix[t] * retrofit_cost[tenure_to_cost_key[t]] for t in tenure_to_cost_key
)
age_band_cost_per_home = retrofit_cost["pre_1919_age_band_average"]

print(f"Tenure-weighted retrofit cost (blended across EHS's national tenure-cost figures): GBP {weighted_cost_per_home:,.0f}/home")
print(f"Pre-1919 age-band average retrofit cost (EHS, age-specific): GBP {age_band_cost_per_home:,.0f}/home")
print()
print(
    "These differ by roughly a third because the tenure-weighted figure blends EHS's cost-to-EPC-C "
    "estimates across all ages of stock within each tenure (including cheaper-to-treat cavity-wall "
    "homes built after 1919), while the age-band figure is specific to pre-1919 solid-wall stock -- "
    "this project's actual archetype. The age-band figure is used as the headline below; the "
    "tenure-weighted figure is shown for reference, not silently discarded."
)
print()

headline_retrofit_cost_per_home = age_band_cost_per_home
n_homes = 1000
total_retrofit_cost = headline_retrofit_cost_per_home * n_homes
print(f"Total retrofit cost for {n_homes:,} homes at the headline figure: GBP {total_retrofit_cost:,.0f}")


Tenure-weighted retrofit cost (blended across EHS's national tenure-cost figures): GBP 6,335/home
Pre-1919 age-band average retrofit cost (EHS, age-specific): GBP 10,728/home

These differ by roughly a third because the tenure-weighted figure blends EHS's cost-to-EPC-C estimates across all ages of stock within each tenure (including cheaper-to-treat cavity-wall homes built after 1919), while the age-band figure is specific to pre-1919 solid-wall stock -- this project's actual archetype. The age-band figure is used as the headline below; the tenure-weighted figure is shown for reference, not silently discarded.

Total retrofit cost for 1,000 homes at the headline figure: GBP 10,728,000


## 3. What this capacity would be worth IF it were contractible (it is not, today)

**External review flagged this as a category error worth correcting explicitly, not softening.** Flexibility-service tenders (Piclo Flex and equivalent DNO/DSO procurement) require metered, dispatchable, verifiable demand reduction against a settled baseline. Passive fabric retrofit alone is none of these things. The GBP/kW figure below is a value-of-capability illustration, not a revenue estimate a landlord could actually collect.


In [4]:
# DNO reinforcement / flexibility-service value (PROJECT.md Section 8.5,
# Stage E): apply the illustrative GBP/kW/year rate to the physical
# peak-reduction figure from Notebook 03's Monte Carlo. No double-counting --
# this is applied on top of the already-computed physical result, not
# layered onto a separately-invented demand figure.
#
# CATEGORY-ERROR WARNING (external review): flexibility-service tenders
# (Piclo Flex and equivalent DNO/DSO procurement) require METERED,
# DISPATCHABLE, VERIFIABLE demand reduction against a settled baseline.
# Passive fabric retrofit, on its own, is none of these things -- it has no
# meter, no controllable asset, no dispatch signal. It CANNOT bid into these
# markets as modelled here. The GBP/kW/year figure below is NOT a realistic
# revenue estimate for a landlord -- it is a "what would this capacity be
# worth IF it were contractible" illustration, and is labelled as such
# throughout this section, not just here.
avoided_kw_p10 = population_03["mc_avoided_kw_p10"][0]
avoided_kw_p50 = population_03["mc_avoided_kw_p50"][0]
avoided_kw_p90 = population_03["mc_avoided_kw_p90"][0]

flex_value = cfg["vpp_economics"]["dno_flexibility_value_gbp_per_kw_per_year"]
flex_low, flex_point, flex_high = flex_value["low"], flex_value["point"], flex_value["high"]

value_per_home_low = avoided_kw_p10 * flex_low
value_per_home_point = avoided_kw_p50 * flex_point
value_per_home_high = avoided_kw_p90 * flex_high

print(f"Avoided peak demand (Notebook 03 Monte Carlo, baseline->EPC-C): P10={avoided_kw_p10:.2f} kW, P50={avoided_kw_p50:.2f} kW, P90={avoided_kw_p90:.2f} kW per home")
print()
print(f"IF this capacity were contractible in a flexibility-service tender at GBP {flex_point}/kW/year "
      f"(illustrative point estimate, PROVISIONAL), it would be worth: GBP {value_per_home_point:.0f}/home/year")
print(f"  Wide range across both avoided-kW and GBP/kW uncertainty: GBP {value_per_home_low:.0f} - {value_per_home_high:.0f}/home/year")
print()
print(
    "It is not actually contractible today: passive fabric has no meter, no controllable asset, and "
    "no dispatch signal, so it cannot participate in a metered, verifiable flexibility tender as "
    "currently structured. Making it contractible would require layering a metered, dispatchable "
    "control asset on top of the fabric (e.g. a smart heat-pump controller), which this project has "
    "not costed. The two channels that ARE realistically monetisable today -- avoided/deferred DNO "
    "reinforcement CAPEX (a one-off, distinct from this recurring service-rate figure) and direct "
    "energy bill savings from reduced heat loss -- are both OMITTED from this notebook, not "
    "quantified as zero. See configs/tenure_insulation_assumptions.yml, vpp_economics and omitted."
)
print()

total_annual_value = value_per_home_point * n_homes
print(f"For {n_homes:,} homes, the illustrative contractible-value figure: ~GBP {total_annual_value:,.0f}/year at the point estimate")
print()

simple_payback_years = headline_retrofit_cost_per_home / value_per_home_point
print(f"Even taken at face value, simple payback of the full retrofit cost from this illustrative value alone: {simple_payback_years:,.0f} years")
print(
    "This channel cannot be the primary financial case for this retrofit, even in the best case where "
    "it were somehow contractible -- comfort, health, EPC compliance and carbon savings must carry "
    "the rest, consistent with this project's core thesis that the 'hidden battery' is real but only "
    "partially and unevenly monetisable, and not through this specific market mechanism as it exists "
    "today."
)


Avoided peak demand (Notebook 03 Monte Carlo, baseline->EPC-C): P10=1.53 kW, P50=1.87 kW, P90=2.30 kW per home

IF this capacity were contractible in a flexibility-service tender at GBP 68/kW/year (illustrative point estimate, PROVISIONAL), it would be worth: GBP 127/home/year
  Wide range across both avoided-kW and GBP/kW uncertainty: GBP 31 - 230/home/year

It is not actually contractible today: passive fabric has no meter, no controllable asset, and no dispatch signal, so it cannot participate in a metered, verifiable flexibility tender as currently structured. Making it contractible would require layering a metered, dispatchable control asset on top of the fabric (e.g. a smart heat-pump controller), which this project has not costed. The two channels that ARE realistically monetisable today -- avoided/deferred DNO reinforcement CAPEX (a one-off, distinct from this recurring service-rate figure) and direct energy bill savings from reduced heat loss -- are both OMITTED from this note

## 4. Battery-equivalence framing (led by what was actually delivered, not the theoretical ceiling)

Electrical-equivalent storage = thermal storage (C x comfort-band delta) / COP. External review: lead with what was actually drawn down in the real simulated event, not the nominal capacity -- otherwise the analogy reads as dressing rather than a grounded comparison.


In [5]:
# Battery-equivalence framing (PROJECT.md Section 8.7): thermal storage is
# not literally interchangeable with electrical storage -- state that in the
# same breath as the number, every time, not just in a footnote.
#
# External review point: lead with what was ACTUALLY delivered in the real
# simulated event, not the theoretical nominal capacity -- "you already
# concede only ~3 kWh-equivalent was used in the real event; lead with that
# as the honest capacity and the analogy survives; lead with 12 kWh and it
# reads as dressing." This section now computes delivered kWh first and
# treats it as the headline; nominal capacity is reported second, clearly
# labelled as a theoretical ceiling.
peak_window_str = cfg["cold_snap_event"]["peak_window"]
start_str, end_str = peak_window_str.split("-")
peak_start = int(start_str.split(":")[0])
peak_end = int(end_str.split(":")[0])
peak_hours = peak_end - peak_start

sim_02_epc_c_peak = sim_02.filter(
    (pl.col("envelope_state") == "epc_c_package")
    & (pl.col("hour_index") % 24 >= peak_start)
    & (pl.col("hour_index") % 24 < peak_end)
)
avoided_kw_epc_c_event = sim_02_epc_c_peak["baseline_kw"].mean() - sim_02_epc_c_peak["vpp_kw"].mean()
delivered_kwh_event = avoided_kw_epc_c_event * peak_hours

print(f"HEADLINE capacity figure: actually-delivered flexibility energy for EPC-C fabric during the "
      f"real Dec 2022 event's peak window (Notebook 02, VPP-curtailed vs continuously-heated, same "
      f"fabric): {delivered_kwh_event:.1f} kWh_e per home")
print()

battery_cost = cfg["vpp_economics"]["domestic_battery_installed_cost_gbp_per_kwh"]
batt_low, batt_point, batt_high = battery_cost["low"], battery_cost["point"], battery_cost["high"]

delivered_value_point = delivered_kwh_event * batt_point
print(f"Valued as battery-hardware-equivalent capacity, at GBP {batt_point}/kWh installed (GROUNDED market range): "
      f"GBP {delivered_value_point:,.0f}/home for what was actually delivered in this event.")
print()

delta_t_usable_k = comfort["preheat_ceiling_c"] - comfort["minimum_c"]
nominal_thermal_storage_kwh = c_kwh_per_k * delta_t_usable_k
electrical_equivalent_kwh = nominal_thermal_storage_kwh / cop
utilisation_fraction = delivered_kwh_event / electrical_equivalent_kwh

print(f"For context, the THEORETICAL CEILING (not the headline): nominal thermal storage is "
      f"C={c_kwh_per_k} kWh/K x usable band {delta_t_usable_k}K = {nominal_thermal_storage_kwh:.0f} kWh_thermal, "
      f"or {electrical_equivalent_kwh:.1f} kWh_e once divided by COP={cop}.")
print(f"Only {utilisation_fraction:.0%} of that nominal ceiling was actually drawn down during this "
      f"specific, comparatively mild peak window -- the gap is the difference between what the "
      f"building's mass could in principle hold and what a 4-hour curtailment against this particular "
      f"event actually calls on. Leading with the nominal figure instead of the delivered one would "
      f"overstate the capacity by roughly 4x for this event.")
print()

battery_equiv_value_low = electrical_equivalent_kwh * batt_low
battery_equiv_value_point = electrical_equivalent_kwh * batt_point
battery_equiv_value_high = electrical_equivalent_kwh * batt_high

print(f"Nominal-ceiling battery-equivalent value (theoretical, not the headline): GBP {battery_equiv_value_point:,.0f}/home "
      f"(range GBP {battery_equiv_value_low:,.0f} - {battery_equiv_value_high:,.0f}/home)")
print()
print(
    "This nominal capacity is the same for baseline and EPC-C fabric -- both share the same building "
    "mass. What retrofit actually changes is how much of it is USABLE during a real curtailment "
    "before the comfort floor is breached (Notebooks 01-03): leaky fabric loses most of this capacity "
    "to heat loss before it can be drawn down on demand -- which is exactly why the DELIVERED figure "
    "above, not the nominal one, is this section's headline."
)


HEADLINE capacity figure: actually-delivered flexibility energy for EPC-C fabric during the real Dec 2022 event's peak window (Notebook 02, VPP-curtailed vs continuously-heated, same fabric): 2.2 kWh_e per home

Valued as battery-hardware-equivalent capacity, at GBP 650/kWh installed (GROUNDED market range): GBP 1,447/home for what was actually delivered in this event.

For context, the THEORETICAL CEILING (not the headline): nominal thermal storage is C=10 kWh/K x usable band 3K = 30 kWh_thermal, or 12.0 kWh_e once divided by COP=2.5.
Only 19% of that nominal ceiling was actually drawn down during this specific, comparatively mild peak window -- the gap is the difference between what the building's mass could in principle hold and what a 4-hour curtailment against this particular event actually calls on. Leading with the nominal figure instead of the delivered one would overstate the capacity by roughly 4x for this event.

Nominal-ceiling battery-equivalent value (theoretical, not the

## 5. Modelling Prose check (PROJECT.md Section 8.3)

In [6]:
headline = (
    f"Retrofitting a pre-1919 solid-wall terrace to EPC-C band unlocks a median {avoided_kw_p50:.2f} kW "
    f"of avoided evening-peak (16:00-20:00) electrical demand per home (Notebook 03 Monte Carlo P50, "
    f"baseline to EPC-C). IF that capacity were contractible in a flexibility-service tender -- which "
    f"it is not, today, since passive fabric has no meter or dispatch signal -- it would be worth "
    f"roughly GBP {value_per_home_point:.0f}/home/year (GBP {flex_point}/kW/year, PROVISIONAL); even "
    f"taken at face value that would need {simple_payback_years:,.0f} years to repay the GBP "
    f"{headline_retrofit_cost_per_home:,.0f}/home retrofit cost (EHS 2024-25 age-band average for this "
    f"archetype), so this channel cannot be the financial case. What the retrofit actually delivered "
    f"during the real December 2022 event -- {delivered_kwh_event:.1f} kWh_e per home of curtailed "
    f"demand -- is worth about GBP {delivered_value_point:,.0f}/home in battery-hardware-equivalent "
    f"terms; the building's theoretical nominal capacity is roughly 4x higher but mostly unused by "
    f"this specific, comparatively mild event. This covers the physical and hardware-comparison "
    f"channels only -- comfort, health, EPC compliance, carbon benefits, avoided reinforcement capex, "
    f"and direct energy-bill savings are all separate and un-monetised in this notebook. All GBP/kW "
    f"and GBP/kWh figures are PROVISIONAL or GROUNDED-but-illustrative (see "
    f"configs/tenure_insulation_assumptions.yml, vpp_economics section), and no settled payment "
    f"mechanism for this exact residential flexibility product currently exists at scale "
    f"(FORWARD-LOOKING)."
)
print(headline)
print()
print("Modelling Prose check (PROJECT.md Section 8.3 pattern):")
print(f"  Number/Unit:    {avoided_kw_p50:.2f} kW/home avoided peak demand; {delivered_kwh_event:.1f} kWh_e/home actually delivered; GBP {headline_retrofit_cost_per_home:,.0f}/home retrofit cost; GBP {delivered_value_point:,.0f}/home delivered-capacity battery-equivalent value")
print("  Denominator:    per home; per year for the illustrative flexibility-service figure only, one-off for retrofit cost and battery-equivalent values")
print("  Mechanism:      fabric retrofit slows heat loss -> longer safe curtailment during evening peak -> avoided grid demand, some of it in principle contractible if metered")
print("  Scope boundary: physical and hardware-comparison channels only; excludes comfort, health, EPC-compliance, carbon benefits, reinforcement capex, and bill savings")
print("  Caveat:         passive fabric cannot itself bid into metered flexibility markets today; DNO GBP/kW and battery GBP/kWh figures are illustrative; no settled residential flexibility payment mechanism exists at scale yet (FORWARD-LOOKING)")


Retrofitting a pre-1919 solid-wall terrace to EPC-C band unlocks a median 1.87 kW of avoided evening-peak (16:00-20:00) electrical demand per home (Notebook 03 Monte Carlo P50, baseline to EPC-C). IF that capacity were contractible in a flexibility-service tender -- which it is not, today, since passive fabric has no meter or dispatch signal -- it would be worth roughly GBP 127/home/year (GBP 68/kW/year, PROVISIONAL); even taken at face value that would need 84 years to repay the GBP 10,728/home retrofit cost (EHS 2024-25 age-band average for this archetype), so this channel cannot be the financial case. What the retrofit actually delivered during the real December 2022 event -- 2.2 kWh_e per home of curtailed demand -- is worth about GBP 1,447/home in battery-hardware-equivalent terms; the building's theoretical nominal capacity is roughly 4x higher but mostly unused by this specific, comparatively mild event. This covers the physical and hardware-comparison channels only -- comfort, 

## 6. Save to `data/intermediate/`

In [7]:
summary = pl.DataFrame([{
    "headline_retrofit_cost_gbp_per_home": headline_retrofit_cost_per_home,
    "tenure_weighted_retrofit_cost_gbp_per_home": weighted_cost_per_home,
    "avoided_kw_p10": avoided_kw_p10,
    "avoided_kw_p50": avoided_kw_p50,
    "avoided_kw_p90": avoided_kw_p90,
    "dno_flex_value_gbp_per_home_per_year_point": value_per_home_point,
    "dno_flex_value_gbp_per_home_per_year_low": value_per_home_low,
    "dno_flex_value_gbp_per_home_per_year_high": value_per_home_high,
    "simple_payback_years": simple_payback_years,
    "nominal_thermal_storage_kwh": nominal_thermal_storage_kwh,
    "electrical_equivalent_kwh": electrical_equivalent_kwh,
    "battery_equivalent_value_gbp_per_home_point": battery_equiv_value_point,
    "delivered_value_gbp_per_home_point": delivered_value_point,
    "delivered_kwh_event_epc_c": delivered_kwh_event,
    "utilisation_fraction": utilisation_fraction,
}])

out_path = Path("../data/intermediate/04_vpp_economics.parquet")
out_path.parent.mkdir(parents=True, exist_ok=True)
summary.write_parquet(out_path)
print(f"Wrote {out_path} ({summary.height} rows)")
summary


Wrote ../data/intermediate/04_vpp_economics.parquet (1 rows)


headline_retrofit_cost_gbp_per_home,tenure_weighted_retrofit_cost_gbp_per_home,avoided_kw_p10,avoided_kw_p50,avoided_kw_p90,dno_flex_value_gbp_per_home_per_year_point,dno_flex_value_gbp_per_home_per_year_low,dno_flex_value_gbp_per_home_per_year_high,simple_payback_years,nominal_thermal_storage_kwh,electrical_equivalent_kwh,battery_equivalent_value_gbp_per_home_point,delivered_value_gbp_per_home_point,delivered_kwh_event_epc_c,utilisation_fraction
i64,f64,f64,f64,f64,f64,f64,f64,f64,i64,f64,f64,f64,f64,f64
10728,6335.42,1.527288,1.867537,2.299703,126.992508,30.545763,229.970298,84.477424,30,12.0,7800.0,1446.512903,2.225404,0.18545
